# Task 5 models v7 — peer-relative change and bot regularity

Two separate unlabeled hypotheses. Fit statistics use the reported lecturer exception for Task 5's test-only file; the written rule's conflict is recorded in the research note. No anomaly labels, F3, or public score enter model fitting or candidate design. The existing v17 rank is frozen. Scores here are rank stability and descriptive profiles only.

In [1]:
from pathlib import Path
import sys, json, joblib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.model_selection import train_test_split
from IPython.display import display
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task5.behavior_models import PeerChangeScorer, BotRegularityScorer
SEED = set_seed()
DATA = "eda_behavior_views_v3"
card = json.loads((SRC / "task5/data/eda" / f"{DATA}.json").read_text(encoding="utf-8"))
assert set(card["splits"]) == {"test"}
frame = pd.read_parquet(SRC / card["splits"]["test"]["path"])
assert frame.player_id.is_unique and "is_anomaly" not in frame
X = frame.drop(columns="player_id")
assert X.shape == (25889, 167)
rows = np.arange(len(X))
fit_idx, hold_idx = train_test_split(rows, train_size=12000, test_size=4000, random_state=SEED)
alt_idx = np.random.default_rng(SEED).choice(fit_idx, size=10000, replace=False)
print("Dataset", DATA, "fit", len(fit_idx), "alternate", len(alt_idx), "unlabeled holdout", len(hold_idx))

Dataset eda_behavior_views_v3 fit 12000 alternate 10000 unlabeled holdout 4000


In [2]:
CHANGE_FAMILIES = ["login_count", "location_changes", "device_count", "ip_hash_entropy",
    "login_time_variance", "session_duration_mean", "play_pattern_consistency", "activity_level",
    "skill_rating", "reaction_time_mean", "purchase_frequency", "purchase_amount_mean",
    "currency_transfers", "communication_frequency"]
CHANGE_COLS = [f"{family}_endpoint_surprise" for family in CHANGE_FAMILIES] + ["login_move_max_km"]
COHORT_COLS = ["account_age_days", "login_count_median4", "activity_level_median4",
               "purchase_frequency_median4", "skill_rating_median4"]
ACTIVITY_COLS = ["login_count_median4", "activity_level_median4"]
REG_HIGH = ["play_pattern_consistency_mean4", "performance_stability_mean4", "behavior_stability"]
REG_LOW = ["login_count_cv4", "activity_level_cv4", "login_time_variance_mean4",
           "session_duration_std_mean4", "reaction_time_std_mean4"]
assert all(c in X for c in CHANGE_COLS + COHORT_COLS + ACTIVITY_COLS + REG_HIGH + REG_LOW)
print("Peer change features", len(CHANGE_COLS), "cohort features", len(COHORT_COLS))
print("Bot activity", ACTIVITY_COLS, "regularity high", REG_HIGH, "regularity low", REG_LOW)

Peer change features 15 cohort features 5
Bot activity ['login_count_median4', 'activity_level_median4'] regularity high ['play_pattern_consistency_mean4', 'performance_stability_mean4', 'behavior_stability'] regularity low ['login_count_cv4', 'activity_level_cv4', 'login_time_variance_mean4', 'session_duration_std_mean4', 'reaction_time_std_mean4']


In [3]:
def top_overlap(a, b, k):
    aa = set(np.lexsort((np.arange(len(a)), -np.asarray(a)))[:k])
    bb = set(np.lexsort((np.arange(len(b)), -np.asarray(b)))[:k])
    return len(aa & bb) / len(aa | bb)
SAVE = SRC / "task5/save"

def saved_or_fit(name, factory, idea):
    paths = [SAVE / f"{name}.joblib", SAVE / f"{name}.json", SAVE / f"{name}-evaluation.json"]
    if any(p.exists() for p in paths):
        assert all(p.exists() for p in paths), f"Incomplete artifact: {name}"
        model_card = json.loads(paths[1].read_text(encoding="utf-8"))
        assert model_card["data_name"] == DATA
        return joblib.load(paths[0]), json.loads(paths[2].read_text(encoding="utf-8"))
    model = factory().fit(X.iloc[fit_idx])
    alternate = factory().fit(X.iloc[alt_idx])
    primary_score = model.anomaly_score(X.iloc[hold_idx])
    alternate_score = alternate.anomaly_score(X.iloc[hold_idx])
    stability = top_overlap(primary_score, alternate_score, k=1000)
    saved = save_model(task=5, name=name, model=model, data_name=DATA,
        card={"idea": idea, "seed": SEED, "fit_rows": len(fit_idx),
              "scope": "reported lecturer exception for unlabeled Task 5 fitting",
              "feature_notebook": "task5/eda/build_eda_behavior_v3.ipynb"})
    evaluation = evaluate_model(task=5, name=name, score=float(stability),
        metric_name="top25_rank_stability_jaccard", higher_is_better=True,
        card={"meaning": "Top-1000 overlap Jaccard on 4000 unlabeled holdout rows across overlapping 12000/10000-row fits; not F3",
              "seed": SEED, "fit_rows": len(fit_idx), "alternate_rows": len(alt_idx), "holdout_rows": len(hold_idx)})
    return model, evaluation

In [4]:
peer_name = "t5_peer_change_v3"
peer, peer_evaluation = saved_or_fit(peer_name,
    lambda: PeerChangeScorer(CHANGE_COLS, COHORT_COLS, neighbors=128, chunk_size=512),
    "Endpoint behavior changes ranked against nearby accounts in stable baseline cohorts")
print("Peer change top-25% rank stability Jaccard (not F3):", peer_evaluation["score"])

Peer change top-25% rank stability Jaccard (not F3): 0.8587360594795539


In [5]:
bot_name = "t5_bot_regular_v3"
bot, bot_evaluation = saved_or_fit(bot_name,
    lambda: BotRegularityScorer(ACTIVITY_COLS, REG_HIGH, REG_LOW),
    "High sustained activity and mechanical regularity must occur together")
print("Bot regularity top-25% rank stability Jaccard (not F3):", bot_evaluation["score"])

Bot regularity top-25% rank stability Jaccard (not F3): 0.9940179461615155


In [6]:
# Reconstruct the frozen v17 ranking exactly from its two saved source models.
old_card = json.loads((SRC / "task5/data/eda/eda_1_v2.json").read_text(encoding="utf-8"))
old = pd.read_parquet(SRC / old_card["splits"]["test"]["path"])
assert old.player_id.equals(frame.player_id)
old_X = old.drop(columns="player_id")
ocsvm = joblib.load(SRC / "task5/save/t5_r2_ocsvm_nu10_engineered_v2.joblib")
gmm = joblib.load(SRC / "task5/save/t5_r2_gmm_engineered_v2.joblib")
def percentile_rank(values):
    return rankdata(np.asarray(values, dtype=float), method="average") / len(values)
base_rank = .5 * percentile_rank(ocsvm.anomaly_score(old_X)) + .5 * percentile_rank(gmm.anomaly_score(old_X))
N_ALERTS = 6531
def fixed_top_k(score):
    selected = np.lexsort((np.arange(len(score)), -np.asarray(score)))[:N_ALERTS]
    result = np.zeros(len(score), dtype=np.int8)
    result[selected] = 1
    return result
base_pred = fixed_top_k(base_rank)
v17 = pd.read_csv(SRC / "submission/kaggle_t12345_t3log_t5_rankblend25_v17.csv")
assert np.array_equal(base_pred, v17.task5.to_numpy(dtype=np.int8))
print("Reconstructed v17 exactly; alerts", int(base_pred.sum()))

Reconstructed v17 exactly; alerts 6531


In [7]:
peer_score = peer.anomaly_score(X)
bot_score = bot.anomaly_score(X)
assert np.isfinite(peer_score).all() and np.isfinite(bot_score).all()
# Frozen before viewing labels or a new leaderboard score. Candidate 2 tests bot's incremental contribution.
h1_rank = .5 * base_rank + .5 * percentile_rank(peer_score)
h2_rank = .8 * h1_rank + .2 * percentile_rank(bot_score)
h1_pred = fixed_top_k(h1_rank)
h2_pred = fixed_top_k(h2_rank)
for name, pred, reference in [("H1 peer change", h1_pred, base_pred), ("H2 bot added", h2_pred, h1_pred)]:
    print(name, "alerts", int(pred.sum()), "swapped in", int(((pred==1)&(reference==0)).sum()),
          "swapped out", int(((pred==0)&(reference==1)).sum()),
          "positive-set Jaccard", round(((pred==1)&(reference==1)).sum()/((pred==1)|(reference==1)).sum(),4))
print("Spearman scores: peer-v17", round(spearmanr(peer_score, base_rank).statistic,3),
      "bot-H1", round(spearmanr(bot_score, h1_rank).statistic,3))

H1 peer change alerts 6531 swapped in 1628 swapped out 1628 positive-set Jaccard 0.6009
H2 bot added alerts 6531 swapped in 887 swapped out 887 positive-set Jaccard 0.7609
Spearman scores: peer-v17 0.509 bot-H1 -0.024


In [8]:
# Descriptive model analysis: features are not anomaly labels.
flags = ["password_changed", "vpn_usage", "suspicious_login_time", "mass_item_sale"]
any_flag = X[flags].gt(0).any(axis=1).to_numpy()
profile_cols = ["login_move_max_km", "login_count_endpoint_surprise", "device_count_endpoint_surprise",
                "skill_rating_endpoint_surprise", "currency_transfers_endpoint_surprise",
                "login_count_cv4", "activity_level_cv4", "play_pattern_consistency_mean4",
                "behavior_stability"]
def profile(mask):
    return {"rows": int(mask.sum()), "any_security_flag_pct": round(100*any_flag[mask].mean(),2),
            **{c: round(float(X.loc[mask,c].median()),3) for c in profile_cols}}
profiles = pd.DataFrame({
    "all": profile(np.ones(len(X),dtype=bool)),
    "v17_alerts": profile(base_pred==1),
    "H1_only": profile((h1_pred==1)&(base_pred==0)),
    "v17_only": profile((h1_pred==0)&(base_pred==1)),
    "H2_only": profile((h2_pred==1)&(h1_pred==0)),
    "H1_only_removed_by_H2": profile((h2_pred==0)&(h1_pred==1)),
}).T
display(profiles)
for col in ["location_changes_endpoint_surprise", "login_count_endpoint_surprise", "currency_transfers_endpoint_surprise", "login_count_cv4"]:
    top = X[col] >= X[col].quantile(.99) if col != "login_count_cv4" else X[col] <= X[col].quantile(.01)
    print(col, "tail accounts", int(top.sum()), "v17/H1/H2 alert rates",
          [round(float(pred[top].mean()),3) for pred in [base_pred,h1_pred,h2_pred]])
print("These are alert-composition diagnostics; no precision, recall, or F3 is inferable.")

,rows,any_security_flag_pct,login_move_max_km,login_count_endpoint_surprise,device_count_endpoint_surprise,skill_rating_endpoint_surprise,currency_transfers_endpoint_surprise,login_count_cv4,activity_level_cv4,play_pattern_consistency_mean4,behavior_stability
all,25889.0,25.44,116.193,7.247,1.0,58.089,0.909,0.124,0.060,0.749,0.822
v17_alerts,6531.0,35.61,122.148,7.621,1.0,63.985,1.113,0.128,0.062,0.752,0.818
H1_only,1628.0,23.53,122.803,7.680,1.0,62.487,1.095,0.126,0.064,0.761,0.826
v17_only,1628.0,32.00,111.043,7.056,1.0,55.013,0.804,0.125,0.061,0.767,0.815
H2_only,887.0,26.16,119.246,9.359,1.0,61.005,0.945,0.113,0.059,0.791,0.849
H1_only_removed_by_H2,887.0,30.21,115.920,5.622,1.0,61.864,1.037,0.136,0.059,0.737,0.800


location_changes_endpoint_surprise tail accounts 259 v17/H1/H2 alert rates [1.0, 1.0, 1.0]
login_count_endpoint_surprise tail accounts 259 v17/H1/H2 alert rates [0.498, 0.579, 0.61]
currency_transfers_endpoint_surprise tail accounts 259 v17/H1/H2 alert rates [1.0, 1.0, 1.0]
login_count_cv4 tail accounts 259 v17/H1/H2 alert rates [0.22, 0.212, 0.224]
These are alert-composition diagnostics; no precision, recall, or F3 is inferable.


In [9]:
# Explain the new peer score by its three strongest peer-relative change dimensions.
def peer_explanations(mask):
    part = X.loc[mask]
    cohort = peer.cohort_scaler_.transform(peer.cohort_imputer_.transform(part[peer.cohort_columns_]))
    changes = peer.change_imputer_.transform(part[peer.change_columns_])
    indexes = peer.neighbors_.kneighbors(cohort, return_distance=False)
    reference = peer.reference_changes_[indexes]
    center = np.median(reference, axis=1)
    spread = np.percentile(reference, 75, axis=1) - np.percentile(reference, 25, axis=1)
    z = np.maximum(changes-center, 0) / (spread + peer.global_floor_)
    top = np.argsort(z, axis=1)[:, -3:]
    counts = np.bincount(top.reshape(-1), minlength=len(peer.change_columns_))
    return pd.Series(counts, index=peer.change_columns_).sort_values(ascending=False)
added_explain = peer_explanations((h1_pred==1)&(base_pred==0))
removed_explain = peer_explanations((h1_pred==0)&(base_pred==1))
display(pd.DataFrame({"H1_added_top3_count": added_explain,
                      "v17_removed_top3_count": removed_explain}).sort_values("H1_added_top3_count", ascending=False).head(15))
print("A top-three contribution describes this score, not an anomaly cause or label.")

,H1_added_top3_count,v17_removed_top3_count
session_duration_mean_endpoint_surprise,450,336
purchase_amount_mean_endpoint_surprise,443,408
location_changes_endpoint_surprise,426,386
currency_transfers_endpoint_surprise,424,371
login_time_variance_endpoint_surprise,422,324
communication_frequency_endpoint_surprise,418,332
login_move_max_km,335,375
reaction_time_mean_endpoint_surprise,327,307
skill_rating_endpoint_surprise,317,330
play_pattern_consistency_endpoint_surprise,277,306


A top-three contribution describes this score, not an anomaly cause or label.


## Model interpretation and decision

The peer-relative score provides a distinct ranking (Spearman 0.509 with v17) but has only 0.85874 top-25% fit-stability Jaccard and causes a large 1,628-in/1,628-out alert swap. Its main added-account dimensions are endpoint changes in session duration, purchase amount, location changes, currency transfers, and login-time variance. The bot score is nearly orthogonal to H1 (Spearman −0.024) and its added accounts show higher play-pattern/behavior stability and lower login-count CV than the removed accounts. This confirms intended score behavior, not anomaly quality. Keep both fixed candidates for controlled comparisons; do not infer F3, precision, recall, or hidden threat labels. The peer cohort medians use all four periods, so an endpoint anomaly can slightly influence cohort placement; the median is used to reduce that effect. A substantial public-score change would still require caution because the public leaderboard has already informed earlier Task 5 decisions.

The bot score does not explicitly model gradual directional drift. The added and removed groups have similar median trend coherence across several families, so this candidate tests sustained activity plus regularity only. A later drift-specific hypothesis would need its own representation and controlled comparison.